# PARC2026 — 76 M3 guarded benchmark training (A100)

Notebook 74のforward/reverse smokeが両方PASSした後に、M3本番trainingを **1 order × 1 trackずつ明示的に** 実行します。

- `equal_data`: 各モデル exactly 4,800 samples / 150 optimizer updates / effective batch 32
- `equal_wall`: 各モデル training-loop 1,800秒以上、最初の安全なoptimizer boundaryで停止
- `forward`: π0.5 → SmolVLA → OpenVLA-OFT
- `reverse`: OpenVLA-OFT → SmolVLA → π0.5

`EXECUTE_BENCHMARK = False` が既定です。Run allではbenchmark trainingを開始しません。Simulator評価・promotionもこのNotebookでは実行しません。


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_runner'
PIN = '447d7f40de86c2a4b6f0bef5b8e7891dd6f66f17'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('76 benchmark code:', got, flush=True)
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
print('GPU:', gpu, flush=True)
if 'A100' not in gpu:
    raise RuntimeError('Notebook 76 requires an NVIDIA A100 runtime')

DRIVE = Path('/content/drive/MyDrive/parc2026-cache')
RUN_ROOT = DRIVE / 'model-benchmark-v1'
DATASET = DRIVE / 'datasets/lerobot_libero_plus_v3_train'
PLAN = RUN_ROOT / 'm3_training_adapter_preflight.json'
SMOKE = RUN_ROOT / 'm3_smoke_summary.json'
STREAMING = DRIVE / 'openvla-streaming-selected-v1/streaming_bridge_contract.json'
EXPECTED_HASH = '73ed0d3b0c5e73c745c0aa2e81517ce1fa40240c75f9eb040d65b6876ba08239'
manifest_candidates = [
    DRIVE / 'pi05-ablation-group-aware-v2/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json',
    ROOT / 'outputs/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json',
]
MANIFEST = None
for candidate in manifest_candidates:
    if not candidate.is_file():
        continue
    data = json.loads(candidate.read_text(encoding='utf-8'))
    if data.get('episode_ids_sha256') == EXPECTED_HASH:
        MANIFEST = candidate
        break
if MANIFEST is None:
    raise FileNotFoundError('Exact D10 manifest not found. Do not regenerate or reselect it.')
for required in (DATASET / 'meta/info.json', PLAN, SMOKE, STREAMING):
    if not required.is_file():
        raise FileNotFoundError(required)
smoke = json.loads(SMOKE.read_text(encoding='utf-8'))
if smoke.get('status') != 'READY_FOR_M3_BENCHMARK':
    raise RuntimeError(f'M3 smoke gate is not ready: {smoke.get("status")}')
if smoke.get('selected_episode_ids_sha256') != EXPECTED_HASH:
    raise RuntimeError('M3 smoke gate D10 hash mismatch')
if smoke.get('benchmark_training_started') is not False:
    raise RuntimeError('Smoke summary unexpectedly claims benchmark training started')
plan = json.loads(PLAN.read_text(encoding='utf-8'))
streaming = json.loads(STREAMING.read_text(encoding='utf-8'))
if plan.get('selected_episode_ids_sha256') != EXPECTED_HASH:
    raise RuntimeError('M3 adapter plan D10 hash mismatch')
if streaming.get('status') != 'PASS' or streaming.get('bridge_type') != 'lerobot_streaming':
    raise RuntimeError('69c streaming contract is not PASS')
if streaming.get('storage_policy', {}).get('full_rlds_materialized') is not False:
    raise RuntimeError('Notebook 76 refuses full RLDS materialization')
WORK_ROOT = ROOT
print(json.dumps({
    'status': 'READY_FOR_EXPLICIT_M3_BENCHMARK_RUN',
    'orders': plan['orders'],
    'equal_data_samples': 4800,
    'equal_data_optimizer_updates': 150,
    'equal_wall_train_loop_sec': 1800,
    'effective_batch_size': 32,
    'automatic_benchmark_start': False,
}, indent=2), flush=True)


## Execute exactly one benchmark sequence
`ORDER` と `TRACK` を選び、実行する回だけ `EXECUTE_BENCHMARK=True` にします。完了済みの同一immutable runは `--resume-completed` で再利用できます。


In [ ]:
ORDER = 'forward'       # 'forward' or 'reverse'
TRACK = 'equal_data'    # 'equal_data' or 'equal_wall'
EXECUTE_BENCHMARK = False  # 本番を開始するときだけ True
if ORDER not in {'forward', 'reverse'}:
    raise ValueError(ORDER)
if TRACK not in {'equal_data', 'equal_wall'}:
    raise ValueError(TRACK)
if not EXECUTE_BENCHMARK:
    print(f'Benchmark NOT started: {ORDER}/{TRACK}. Set EXECUTE_BENCHMARK=True explicitly.', flush=True)
else:
    env = os.environ.copy()
    env['PARC_M3_EXECUTE'] = '1'
    subprocess.run([
        sys.executable, '-u', '-m', 'tools.benchmark.run_m3_guarded',
        '--repo-root', str(REPO), '--adapter-plan', str(PLAN),
        '--manifest', str(MANIFEST), '--dataset-root', str(DATASET),
        '--streaming-contract', str(STREAMING), '--work-root', str(WORK_ROOT),
        '--run-root', str(RUN_ROOT), '--mode', 'benchmark', '--order', ORDER,
        '--track', TRACK, '--resume-completed',
    ], cwd=str(REPO), env=env, check=True)
    status_path = RUN_ROOT / f'runs/{ORDER}/{TRACK}/orchestrator_status.json'
    status = json.loads(status_path.read_text(encoding='utf-8'))
    if status.get('status') != 'PASS':
        raise RuntimeError(status)
    print(f'=== 76 M3 BENCHMARK TRAINING {ORDER}/{TRACK}: PASS ===', flush=True)
    print(json.dumps(status, indent=2), flush=True)
    print('Simulator evaluation is still pending; no promotion has been made.', flush=True)


## Training matrix gate
4 sequence（forward/reverse × equal_data/equal_wall）が全部揃ったときだけ、simulator評価へ進めるsummaryを作ります。training lossだけではpromotionしません。


In [ ]:
models = ['pi05', 'smolvla', 'openvla_oft']
orders = ['forward', 'reverse']
tracks = ['equal_data', 'equal_wall']
records = []
missing = []
for order in orders:
    for track in tracks:
        status_path = RUN_ROOT / f'runs/{order}/{track}/orchestrator_status.json'
        if not status_path.is_file():
            missing.append(str(status_path))
            continue
        status = json.loads(status_path.read_text(encoding='utf-8'))
        if status.get('status') != 'PASS':
            raise RuntimeError(f'{order}/{track} orchestration not PASS: {status}')
        for model in models:
            path = RUN_ROOT / f'runs/{order}/{track}/{model}/training_result.json'
            if not path.is_file():
                missing.append(str(path))
                continue
            r = json.loads(path.read_text(encoding='utf-8'))
            if r.get('status') != 'PASS' or r.get('mode') != 'benchmark':
                raise RuntimeError(r)
            if r.get('selected_episode_ids_sha256') != EXPECTED_HASH:
                raise RuntimeError(f'{order}/{track}/{model}: D10 hash mismatch')
            if int(r.get('effective_batch_size', -1)) != 32:
                raise RuntimeError(f'{order}/{track}/{model}: effective batch drift')
            if track == 'equal_data':
                if int(r.get('samples_consumed', -1)) != 4800 or int(r.get('optimizer_updates', -1)) != 150:
                    raise RuntimeError(f'{order}/{model}: equal-data target mismatch')
            else:
                if float(r.get('train_wall_time', -1)) < 1800.0:
                    raise RuntimeError(f'{order}/{model}: equal-wall target not reached')
            records.append(r)
if missing:
    print('M3 training matrix is not complete yet. Missing:', flush=True)
    for path in missing:
        print(' -', path, flush=True)
else:
    if len(records) != 12:
        raise RuntimeError(f'Expected 12 M3 training results, got {len(records)}')
    for track in tracks:
        hashes = {r['sampling_schedule_sha256'] for r in records if r['track'] == track}
        if len(hashes) != 1:
            raise RuntimeError(f'{track}: sampling schedule SHA differs across models/orders: {hashes}')
    summary = {
        'schema_version': 1,
        'stage': 'M3_guarded_training_matrix',
        'status': 'READY_FOR_M3_SIMULATOR_EVALUATION',
        'selected_dataset_variant': 'V2_SQRT_BALANCED_RAW',
        'selected_episode_ids_sha256': EXPECTED_HASH,
        'orders': orders,
        'tracks': tracks,
        'models': models,
        'result_count': len(records),
        'equal_data_samples': 4800,
        'equal_data_optimizer_updates': 150,
        'equal_wall_train_loop_sec_min': 1800,
        'effective_batch_size': 32,
        'training_loss_used_for_promotion': False,
        'simulator_evaluation_completed': False,
        'promotion_completed': False,
    }
    out = RUN_ROOT / 'm3_training_matrix_summary.json'
    out.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(summary, indent=2), flush=True)
    print('=== 76 M3 TRAINING MATRIX: READY_FOR_M3_SIMULATOR_EVALUATION ===', flush=True)
    print('No model has been promoted.', flush=True)
